## Instructions

The below file is used to train our sentiment analysis model.
Due to high computation power if you wish to run the below code we suggest you run it on colab, or if you have a GPU such as T4 on your device then you can run this locally after modifying the code to fetch data from your local machine.

## Install Required Libraries

This installs or updates the latest versions of essential libraries:

transformers for BERT model support
tensorflow for training the model
pandas for data handling
scikit-learn for dataset splitting and evaluation

In [ ]:
# ✅ Install Transformers, TensorFlow, Pandas, and Scikit-learn
!pip install -U transformers tensorflow pandas scikit-learn --quiet


## Upload Training Data

Prompts you to upload a ZIP containing:
A CSV file with review texts and labels (0 or 1)
Folders pos/ and neg/ with .txt review files

Extracts the ZIP into a /data directory

In [ ]:
from google.colab import files
import zipfile
import os

print("\n📤 Please upload a ZIP file containing:\n- sentiment_data.csv (with 'text' and 'label')\n- pos/ and neg/ folders with .txt reviews")
uploaded = files.upload()

# Unzip uploaded file
for filename in uploaded.keys():
    if filename.endswith(".zip"):
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall("data")
        print(f"✅ Extracted {filename} to /data")

!ls data


## Load and combine CSV and Text Files

Loads and cleans the CSV file (text, label)
Loads .txt files from pos/ and neg/ folders and assigns binary labels
Combines both into one final training dataframe

In [ ]:
import pandas as pd

# Load CSV
csv_path = "data/Train/sentiment trainer.csv"
print(f"\n📥 Loading CSV data from: {csv_path}")
csv_df = pd.read_csv(csv_path)
csv_df = csv_df.dropna(subset=["text", "label"])
csv_df['label'] = csv_df['label'].astype(int)

# Load TXT files
def load_text_files(folder_path):
    texts, labels = [], []
    for label_folder in ["pos", "neg"]:
        label = 1 if label_folder == "pos" else 0
        folder = os.path.join(folder_path, label_folder)
        if not os.path.exists(folder):
            continue
        for filename in os.listdir(folder):
            file_path = os.path.join(folder, filename)
            if filename.endswith(".txt"):
                with open(file_path, encoding='utf-8') as f:
                    texts.append(f.read())
                    labels.append(label)
    return pd.DataFrame({"text": texts, "label": labels})

txt_folder = "data/Train"
txt_df = load_text_files(txt_folder)

# Combine both datasets
df = pd.concat([csv_df, txt_df]).dropna().reset_index(drop=True)
print(f"✅ Total training samples: {len(df)}")


## Tokenize the Data Using DistilBERT Tokenizer

Loads distilbert-base-uncased tokenizer
Tokenizes the text into model-ready input (input IDs + attention mask)
Splits data into train and validation sets (80/20)
Batches and prepares TensorFlow datasets


In [ ]:
from transformers import AutoTokenizer
from sklearn.model_selection import train_test_split
import tensorflow as tf

model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Tokenize texts
def tokenize_texts(texts):
    return tokenizer(
        list(texts),
        max_length=256,
        padding="max_length",
        truncation=True,
        return_tensors="tf"
    )

# Train-test split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    df['text'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42
)

train_encodings = tokenize_texts(train_texts)
val_encodings = tokenize_texts(val_texts)

# Convert to TensorFlow Datasets
train_dataset = tf.data.Dataset.from_tensor_slices((dict(train_encodings), train_labels)).shuffle(1000).batch(16)
val_dataset = tf.data.Dataset.from_tensor_slices((dict(val_encodings), val_labels)).batch(16)


## Define and Train the Model with Keras

Loads DistilBERT for binary classification (num_labels=2)
Sets up optimizer, loss function, and evaluation metric
Trains the model for 3 epochs using Keras' fit() API

In [ ]:
from transformers import TFAutoModelForSequenceClassification

model = TFAutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

optimizer = tf.keras.optimizers.Adam(learning_rate=2e-5)
loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)
metrics = [tf.keras.metrics.SparseCategoricalAccuracy()]

model.compile(optimizer=optimizer, loss=loss, metrics=metrics)

print("\n🚀 Training BERT model with Keras...")
model.fit(train_dataset, validation_data=val_dataset, epochs=3)


## Save and Download the Model

Saves the trained model (weights and config) and tokenizer
Zips the saved folder
Automatically downloads the ZIP so you can use the model locally or for inference

In [ ]:
model.save_pretrained("bert_sentiment_model")
tokenizer.save_pretrained("bert_sentiment_model")

import shutil
shutil.make_archive("bert_sentiment_model", 'zip', "bert_sentiment_model")
files.download("bert_sentiment_model.zip")
print("✅ Model ready and downloading: bert_sentiment_model.zip")
